[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/18_embedding.ipynb)

# 🟢 Easy: Embedding Layer

Implement an **embedding lookup table** from scratch.

### Signature
```python
class MyEmbedding(nn.Module):
    def __init__(self, num_embeddings: int, embedding_dim: int): ...
    def forward(self, indices: Tensor) -> Tensor: ...
```

### Rules
- `self.weight`: `nn.Parameter` of shape `(num_embeddings, embedding_dim)`
- Forward: index into weight matrix — `weight[indices]`
- Do NOT use `nn.Embedding`

### Embeddingとは
- ディープラーニングにおけるエンベッディング（埋め込み表現）とは、
単語、画像、ユーザーIDなどの「離散的で人間が理解しやすいデータ」を、
コンピュータが計算・処理しやすい「意味を持った数値のベクトル（配列）」に
変換する技術のことです。
- エンベッディングの基本と仕組みエンベッディング空間（ベクトル空間）では、
データ同士の意味的な関係性が距離として表現されます。
- 意味の数値化: 「犬」と「猫」は近いベクトルを持ち、
「犬」と「車」は遠いベクトルを持ちます。
- 計算の適用: ベクトル同士の足し算や引き算で意味の操作が可能です
  - （例: $\(ベクトル(王) - ベクトル(男) + ベクトル(女) \approx ベクトル(女王)\$)

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [1]:
import torch
import torch.nn as nn

/usr/local/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [2]:
# ✏️ YOUR IMPLEMENTATION HERE

class MyEmbedding(nn.Module):
    def __init__(self, num_embeddings, embedding_dim):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(num_embeddings, embedding_dim))

    def forward(self, indices):
        return self.weight[indices]

### nn.Parameter
`nn.Parameter`で包むと`model.parameters()`に自動登録され、`optimizer`が更新対象として認識する。
|種類           |optimizerが更新|save/loadで保存|
|---------------|---------------|---------------|
|nn.Parameter   | する           | する          |
|register_buffer| しない         | する          |
|ただのテンソル  | しない          | しない        |

### torch.randnによる初期化
重みをランダム値で初期化する理由は以下の通り：
- ゼロ初期化は対称性の問題で学習が進まなくなる(全ニューロンが同じ値・同じ勾配のまま動く)
- Embeddingでは各行を区別する必要があり、同じ値だと全IDが見分けられない

### emb(idx)が動く仕組み
`ebm(idx)`の様にインスタンスを関数の様に実行できるのは`nn.Module`の`__call__`の働きによる。
```python
ebm(idx) -> nn.Module.__call__(emb, idx) -> emb.forward(idx)
```

In [26]:
# 🧪 Debug
emb = MyEmbedding(10, 4)
idx = torch.tensor([0, 3, 7])
print(emb.weight)
print(emb([0, 3, 7])) # __call__でforward()が実行されるようにnn.Moduleで定義ずみ
print(emb.weight[[0, 3, 7]]) # 上と同じことを__call__を使わずに直接記述する
print('Output shape:', emb(idx).shape)
print('Matches manual:', torch.equal(emb(idx)[0], emb.weight[0]))

Parameter containing:
tensor([[-0.4677, -0.0364,  1.4442, -1.1886],
        [ 2.1967, -0.3839,  0.4885, -0.3287],
        [ 0.7428,  0.6159, -0.8329,  1.3126],
        [ 1.3174,  0.1913,  0.3435, -1.4461],
        [-1.7452, -1.0078, -1.0685, -0.7203],
        [-0.0365,  0.3291, -0.4363,  1.3115],
        [ 1.5130,  2.1826,  1.0179, -0.1711],
        [-1.2074, -0.2472,  0.1480,  1.3204],
        [ 0.9018,  0.4822, -1.0018,  0.0937],
        [-0.7467, -0.5528, -1.6004, -0.6154]], requires_grad=True)
tensor([[-0.4677, -0.0364,  1.4442, -1.1886],
        [ 1.3174,  0.1913,  0.3435, -1.4461],
        [-1.2074, -0.2472,  0.1480,  1.3204]], grad_fn=<IndexBackward0>)
tensor([[-0.4677, -0.0364,  1.4442, -1.1886],
        [ 1.3174,  0.1913,  0.3435, -1.4461],
        [-1.2074, -0.2472,  0.1480,  1.3204]], grad_fn=<IndexBackward0>)
Output shape: torch.Size([3, 4])
Matches manual: True


In [4]:
# ✅ SUBMIT
from torch_judge import check
check('embedding')


🧪 Testing: Embedding Layer (Easy)
──────────────────────────────────────────────────
  ✅ [1/4] Weight shape (0.3ms)
  ✅ [2/4] Lookup correctness (0.2ms)
  ✅ [3/4] Batch of indices (0.1ms)
  ✅ [4/4] Gradient flow (9.8ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (10.4ms total)
  Progress saved. Run status() to see your dashboard.

